# 07 — MC dropout calibration

Per `plan.md` §8 + §11 step 8.

* Re-score one fold's test galaxy with MC dropout (T=50).
* Build a reliability diagram, compute ECE.
* If ECE > 0.10, fit temperature scaling on the within-fold val set
  and compare pre/post diagrams.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from hishells.augment import no_augment
from hishells.catalog import LOGO_GALAXIES_19, load_catalog
from hishells.cubes import sigma_rms
from hishells.data import (
    CubeStore,
    DatasetConfig,
    LOGOSplitter,
    ShellWindowDataset,
    make_subset,
)
from hishells.eval import expected_calibration_error
from hishells.model import build_model
from hishells.predict import (
    apply_temperature,
    fit_temperature,
    predict_dataset_mc,
)
from hishells.train import load_checkpoint, predict_dataset
from hishells.windows import NegSampleConfig, build_window_table

In [ ]:
REPO = Path('..').resolve()
ABLATION = 'v1_baseline'
FOLD = 'NGC_2403'  # change me

ckpt_path = REPO / 'results' / 'checkpoints' / ABLATION / f'{FOLD}.pt'
model = build_model('small')
load_checkpoint(ckpt_path, model)
print(f'loaded {ckpt_path.name}')

In [ ]:
cat = load_catalog(REPO / 'Data' / 'J_AJ_141_23')
holes = cat.filter(hole_types=(2, 3), downloaded_in=REPO / 'Data' / 'THINGS')
cubes = CubeStore(REPO / 'Data' / 'THINGS', max_cubes=2)
sigmas = {g: sigma_rms(cubes(g)) for g in LOGO_GALAXIES_19 if g in set(holes['galaxy_id'])}
neg_cfg = NegSampleConfig(ratio=5.0, hard_frac=0.75, rng_seed=0)
table = build_window_table(holes, {g: cubes(g) for g in sigmas}, neg_cfg, cube_sigmas=sigmas)
splitter = LOGOSplitter(table, galaxies=tuple(sigmas.keys()), val_frac=0.10, rng_seed=0)
fold = next(f for f in splitter if f.test_galaxy == FOLD)
ds = ShellWindowDataset(table, cubes, sigma_rms_by_galaxy=sigmas, config=DatasetConfig(window_pix=64, augment=no_augment()))
test_set = make_subset(ds, fold.test_idx)
val_set = make_subset(ds, fold.val_idx)

In [ ]:
mc, test_labels, _ = predict_dataset_mc(model, test_set, T=50, batch_size=64)
print(f'test set: n={len(test_labels)}, mean score={mc.score_mean.mean():.3f}, mean std={mc.score_std.mean():.3f}')

## Reliability diagram

In [ ]:
def reliability_diagram(scores, labels, n_bins=15, ax=None, label=None):
    edges = np.linspace(0, 1, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    bin_idx = np.clip(np.digitize(scores, edges) - 1, 0, n_bins - 1)
    accs = np.zeros(n_bins); confs = np.zeros(n_bins); counts = np.zeros(n_bins)
    for b in range(n_bins):
        m = bin_idx == b
        if m.any():
            accs[b] = labels[m].mean()
            confs[b] = scores[m].mean()
            counts[b] = m.sum()
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.bar(centers, accs, width=1.0/n_bins, edgecolor='k', alpha=0.6, label=label or 'predicted')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('predicted probability'); ax.set_ylabel('empirical accuracy')
    return ax

fig, ax = plt.subplots(figsize=(5, 5))
reliability_diagram(mc.score_mean, test_labels, ax=ax, label='MC mean')
ax.set_title(f'{FOLD}, ECE={expected_calibration_error(mc.score_mean, test_labels):.3f}')

## Temperature scaling rerun

In [ ]:
import torch
from hishells.train import _window_collate
from torch.utils.data import DataLoader

def collect_logits(model, dataset):
    model.eval()
    loader = DataLoader(dataset, batch_size=64, shuffle=False, collate_fn=_window_collate)
    logits = []; labels = []
    with torch.no_grad():
        for xs, ys, _ in loader:
            logits.append(model(xs).flatten().cpu().numpy())
            labels.append(ys.flatten().numpy())
    return np.concatenate(logits), np.concatenate(labels)

val_logits, val_labels = collect_logits(model, val_set)
test_logits, _ = collect_logits(model, test_set)
T = fit_temperature(val_logits, val_labels)
print(f'fitted temperature T = {T:.3f}')
test_scores_t = apply_temperature(test_logits, T)
print(f'ECE pre={expected_calibration_error(mc.score_mean, test_labels):.3f}  post={expected_calibration_error(test_scores_t, test_labels):.3f}')
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
reliability_diagram(mc.score_mean, test_labels, ax=axes[0], label='before')
axes[0].set_title('MC dropout')
reliability_diagram(test_scores_t, test_labels, ax=axes[1], label='after T-scaling')
axes[1].set_title(f'+ T-scaling (T={T:.2f})')

## Uncertainty vs correctness

For MC dropout to be useful as a triage signal, ``score_std`` should
correlate with ``|score_mean - label|`` (i.e. the model is more
uncertain on samples it gets wrong).

In [ ]:
err = np.abs(mc.score_mean - test_labels)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(err, mc.score_std, s=10, alpha=0.4)
ax.set_xlabel('|score_mean - label|')
ax.set_ylabel('MC dropout std')
rho = float(np.corrcoef(err, mc.score_std)[0, 1])
ax.set_title(f'corr={rho:.2f}  (>0 means uncertainty tracks errors)')